# Libraries

In [34]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [5]:
df = pd.read_csv("database\\model_data.csv")
df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'])

In [6]:
df

,FECHA_HORA,IDELEM,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,2017-01-01 00:00:00,3910,129.000000,1.000000,3.750000,40.0
1,2017-01-01 00:00:00,3911,151.000000,0.666667,3.666667,40.0
2,2017-01-01 00:00:00,3913,106.750000,0.000000,3.000000,40.0
3,2017-01-01 00:00:00,3914,145.500000,0.250000,3.750000,40.0
4,2017-01-01 00:00:00,3915,118.250000,0.250000,2.750000,40.0
...,...,...,...,...,...,...
464305,2024-12-31 23:00:00,3913,57.333333,0.000000,1.333333,62.0
464306,2024-12-31 23:00:00,3914,56.500000,0.000000,1.250000,62.0
464307,2024-12-31 23:00:00,3915,43.000000,0.000000,0.666667,62.0
464308,2024-12-31 23:00:00,3917,50.000000,0.000000,1.000000,62.0


In [8]:
# Me interesa FECHA_HORA para separar en entrenamiento y test
df['HORA'] = df['FECHA_HORA'].dt.hour

In [46]:
df

,FECHA_HORA,IDELEM,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION,HORA
0,2017-01-01 00:00:00,3910,129.000000,1.000000,3.750000,40.0,0
1,2017-01-01 00:00:00,3911,151.000000,0.666667,3.666667,40.0,0
2,2017-01-01 00:00:00,3913,106.750000,0.000000,3.000000,40.0,0
3,2017-01-01 00:00:00,3914,145.500000,0.250000,3.750000,40.0,0
4,2017-01-01 00:00:00,3915,118.250000,0.250000,2.750000,40.0,0
...,...,...,...,...,...,...,...
464305,2024-12-31 23:00:00,3913,57.333333,0.000000,1.333333,62.0,23
464306,2024-12-31 23:00:00,3914,56.500000,0.000000,1.250000,62.0,23
464307,2024-12-31 23:00:00,3915,43.000000,0.000000,0.666667,62.0,23
464308,2024-12-31 23:00:00,3917,50.000000,0.000000,1.000000,62.0,23


In [47]:
train = df[df['FECHA_HORA'] < '2024-01-01']
test = df[df['FECHA_HORA'] >= '2024-01-01']

In [48]:
train = train.groupby(['IDELEM','HORA']).agg({
    'INTENSIDAD': 'mean',
    'OCUPACION': 'mean',
    'CARGA': 'mean',
    'VALOR_CONTAMINACION': 'mean'
}).reset_index()

In [49]:
test = test.groupby(['IDELEM','HORA']).agg({
    'INTENSIDAD': 'mean',
    'OCUPACION': 'mean',
    'CARGA': 'mean',
    'VALOR_CONTAMINACION': 'mean'
}).reset_index()

In [50]:
train

,IDELEM,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,3910,0,134.577953,0.803358,5.137581,36.527670
1,3910,1,87.190576,0.540000,3.311276,29.538765
2,3910,2,57.170653,0.304351,1.970520,23.306926
3,3910,3,39.407208,0.224500,1.233250,19.521000
4,3910,4,31.232221,0.215236,0.871319,17.137048
...,...,...,...,...,...,...
187,5547,19,823.303023,3.131838,16.428583,47.818138
188,5547,20,772.313986,2.957476,15.728219,53.545237
189,5547,21,638.662112,2.328859,13.173458,56.586769
190,5547,22,420.098768,1.461732,8.817378,52.053743


In [51]:
test

,IDELEM,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,3910,0,133.928962,0.839936,5.130920,26.639344
1,3910,1,82.498168,0.507555,3.119963,22.538462
2,3910,2,49.528082,0.194521,1.621233,17.745205
3,3910,3,31.731050,0.058904,0.847945,14.961644
4,3910,4,25.061126,0.030907,0.541896,13.065934
...,...,...,...,...,...,...
187,5547,19,784.482735,2.558011,15.496547,35.914365
188,5547,20,728.908840,2.221685,14.493785,38.295580
189,5547,21,585.156768,1.625691,11.902624,40.314917
190,5547,22,367.856814,0.741713,7.472836,36.599448


In [52]:
features = ['IDELEM', 'HORA', 'INTENSIDAD', 'OCUPACION', 'CARGA']
target = ['VALOR_CONTAMINACION']

In [53]:
X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

In [54]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

c:\Users\andre\Documents\Master\TFM\spark_anomaly_detection\.venv\Lib\site-packages\sklearn\base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [55]:
# Save model to pickle
import pickle
with open('database\\models\\pollution_model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [56]:
# Load model
with open('database\\models\\pollution_model.pkl', 'rb') as f:
    model = pickle.load(f)

In [57]:
y_pred = model.predict(X_test)

In [58]:
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

MSE: 107.60
RMSE: 10.37


In [59]:
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")

MAE: 9.83


In [60]:
r2 = r2_score(y_test, y_pred)
print(f"R-cuadrado (R²): {r2:.2f}")

R-cuadrado (R²): -0.88


# Repeat process but SCALING the data

In [61]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scale_columns = ['INTENSIDAD', 'OCUPACION', 'CARGA']
train[scale_columns] = scaler.fit_transform(train[scale_columns])
test[scale_columns] = scaler.fit_transform(test[scale_columns])

In [62]:
train

,IDELEM,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,3910,0,-1.153185,-1.002389,-1.076670,36.527670
1,3910,1,-1.314774,-1.097303,-1.224576,29.538765
2,3910,2,-1.417141,-1.182231,-1.333160,23.306926
3,3910,3,-1.477714,-1.211009,-1.392869,19.521000
4,3910,4,-1.505590,-1.214348,-1.422180,17.137048
...,...,...,...,...,...,...
187,5547,19,1.195341,-0.163203,-0.162249,47.818138
188,5547,20,1.021470,-0.226043,-0.218969,53.545237
189,5547,21,0.565722,-0.452597,-0.425871,56.586769
190,5547,22,-0.179570,-0.765110,-0.778655,52.053743


In [63]:
test

,IDELEM,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,3910,0,-1.132898,-0.914830,-1.037955,26.639344
1,3910,1,-1.310435,-1.030929,-1.199704,22.538462
2,3910,2,-1.424247,-1.140269,-1.320254,17.745205
3,3910,3,-1.485682,-1.187639,-1.382452,14.961644
4,3910,4,-1.508706,-1.197418,-1.407069,13.065934
...,...,...,...,...,...,...
187,5547,19,1.112796,-0.314720,-0.204205,35.914365
188,5547,20,0.920956,-0.432196,-0.284861,38.295580
189,5547,21,0.424728,-0.640372,-0.493279,40.314917
190,5547,22,-0.325385,-0.949139,-0.849585,36.599448


In [64]:
X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

In [65]:
model_scaled = RandomForestRegressor()
model_scaled.fit(X_train, y_train)

c:\Users\andre\Documents\Master\TFM\spark_anomaly_detection\.venv\Lib\site-packages\sklearn\base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [66]:
with open('database\\models\\pollution_model_scaled.pkl', 'wb') as f:
    pickle.dump(model_scaled, f)

In [67]:
with open('database\\models\\pollution_model_scaled.pkl', 'rb') as f:
    model_scaled = pickle.load(f)

In [69]:
y_pred = model_scaled.predict(X_test)

In [70]:
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

MSE: 110.50
RMSE: 10.51


In [71]:
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")

MAE: 10.01


In [72]:
r2 = r2_score(y_test, y_pred)
print(f"R-cuadrado (R²): {r2:.2f}")

R-cuadrado (R²): -0.93


No se si el r2 sirve en regresion, creo que no